# Bayesian Fitting with BUMPS

This notebook demonstrates the full high-level Bayesian MCMC API in
EasyReflectometry.

It covers:

- Building a reflectometry model with realistic bounds (critical for MCMC).
- Classical optimisation first (good starting point for Bayesian sampling).
- High-level DREAM MCMC sampling via
  ``MultiFitter.sample()`` and ``PosteriorResults``.
- Posterior inspection: summary table, corner plot, trace plot, credible
  intervals, Gelman-Rubin R-hat.
- Posterior-predictive checks: reflectivity and SLD profile with 95 %
  credible bands.

**Note**: Requires ``corner`` and ``arviz`` for some plots (install via
``pip install easyreflectometry[bayesian]``).

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import scipp as sc
import matplotlib.pyplot as plt

from easyreflectometry.data.measurement import load
from easyreflectometry.sample import Material, Layer, Multilayer, Sample
from easyreflectometry.model import Model, PercentageFwhm
from easyreflectometry.calculators import CalculatorFactory
from easyreflectometry.fitting import MultiFitter
from easyreflectometry.analysis.bayesian import (
    PosteriorResults,
    posterior_summary,
    plot_corner,
    plot_trace,
    credible_intervals,
    posterior_predictive_reflectivity,
    posterior_predictive_sld_profile,
)
from easyscience.fitting import AvailableMinimizers

print('All libraries imported successfully.')

In [ ]:
# ---- Load experimental data -------------------------------------------------
data_path = 'example.ort'
data = load(data_path)
print('Data loaded with keys:', list(data.keys()))

# Quick look at the data
qz = data['coords']['Qz_0'].values
r_data = data['data']['R_0'].values

plt.figure(figsize=(7, 4))
plt.semilogy(qz, r_data, 'o', label='Data')
plt.xlabel('Q / Å⁻¹')
plt.ylabel('Reflectivity')
plt.title('Experimental data (example.ort)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ---- Create a monolayer model (Si / Film / D₂O) ----------------------------
si   = Material(sld=2.07, isld=0.0, name='Si')
film = Material(sld=2.0,  isld=0.0, name='Film')
d2o  = Material(sld=6.36, isld=0.0, name='D2O')

si_layer   = Layer(material=si,   thickness=0.0,   roughness=3.0, name='Si')
film_layer = Layer(material=film, thickness=250.0, roughness=3.0, name='Film')
d2o_layer  = Layer(material=d2o,  thickness=0.0,   roughness=3.0, name='D2O')

sample = Sample(
    Multilayer(si_layer),
    Multilayer(film_layer),
    Multilayer(d2o_layer),
    name='Monolayer Sample',
)

resolution = PercentageFwhm(0.02)
model = Model(
    sample=sample,
    scale=1.0,
    background=1e-6,
    resolution_function=resolution,
    name='Monolayer Model',
)

# ---- Make key parameters free with realistic bounds (essential for MCMC) -----
film_layer.thickness.fixed = False
film_layer.thickness.bounds = (100, 400)

film.sld.fixed = False
film.sld.bounds = (0.5, 4.0)

model.scale.fixed = False
model.scale.bounds = (0.8, 1.2)

model.background.fixed = False
model.background.bounds = (1e-7, 1e-5)

print('Model created with the following free parameters:')
for p in model.get_parameters():
    if not p.fixed:
        print(f'  {p.name}: value={p.value}, bounds={p.bounds}')

In [ ]:
# ---- Set up the calculator and fitter ---------------------------------------
interface = CalculatorFactory()
interface.switch('refnx')  # or 'refl1d'
model.interface = interface

fitter = MultiFitter(model)
# Use the BUMPS backend — DREAM sampling is only available through BUMPS.
fitter.switch_minimizer(AvailableMinimizers.Bumps)

print('Fitter ready with minimizer:', fitter.easy_science_multi_fitter.minimizer.name)

In [ ]:
# ---- Classical fit first ----------------------------------------------------
# A classical optimisation gives a good starting point and a sanity check
# before launching the (much more expensive) MCMC sampler.

analysed = fitter.fit(data)

print('Classical fit successful:', analysed.get('success', 'N/A'))
print('Reduced χ² ≈', analysed.get('reduced_chi', 'N/A'))

# Plot the fit
r_model = analysed['R_0_model'].values

plt.figure(figsize=(8, 5))
plt.semilogy(qz, r_data, 'o', label='Data', alpha=0.7)
plt.semilogy(qz, r_model, '-', label='Classical BUMPS fit', linewidth=2)
plt.xlabel('Q / Å⁻¹')
plt.ylabel('Reflectivity')
plt.title('Classical fit before Bayesian sampling')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ---- Bayesian MCMC sampling -------------------------------------------------
# ``MultiFitter.sample()`` delegates to the BUMPS DREAM sampler.
# All keyword arguments are forwarded with user-friendly names:
#   ``samples``   ← total retained samples
#   ``burn``      ← burn‑in steps
#   ``thin``      ← thinning interval
#   ``chains``    ← DREAM population count (alias for ``pop``)
#   ``population``← BUMPS‑native ``pop`` for advanced users
#   ``seed``      ← random seed for reproducibility

posterior_dict = fitter.sample(
    data,
    samples=500,   # Short for demo; use 20 k+ in production
    burn=100,
    thin=10,
    seed=42,
)

print('DREAM sampling complete.')
print(f'  Posterior shape  : {posterior_dict["draws"].shape}')
print(f'  Parameters        : {posterior_dict["param_names"]}')

In [ ]:
# ---- Wrap in PosteriorResults -----------------------------------------------
# ``PosteriorResults`` gives you a convenient object with built-in analysis
# methods.  You can also use the raw ``dict`` with the standalone functions
# — both styles are shown below.

posterior = PosteriorResults(
    draws=posterior_dict['draws'],
    param_names=posterior_dict['param_names'],
    logp=posterior_dict.get('logp'),
    sampler_state=posterior_dict.get('state'),
)

print(posterior)

In [ ]:
# ---- Posterior summary table ------------------------------------------------
# ``.summary()`` prints mean, standard deviation and 95 % HDI for each parameter.

print(posterior.summary())
print()

# The standalone function produces the same output:
# print(posterior_summary(posterior.draws, posterior.param_names))

In [ ]:
# ---- Corner plot ------------------------------------------------------------
# Visualise all pairwise parameter correlations and marginal distributions.

posterior.corner()
plt.suptitle('Posterior distributions from BUMPS DREAM', y=1.02)
plt.show()

# Standalone equivalent:
# plot_corner(posterior.draws, posterior.param_names)

In [ ]:
# ---- Trace plot -------------------------------------------------------------
# Inspect the MCMC chains for convergence.  Requires ``arviz``.

try:
    posterior.trace()
    plt.suptitle('MCMC trace plots', y=1.02)
    plt.show()
except ImportError:
    print('Skipped: arviz is not installed.')
    print('Install with: pip install easyreflectometry[bayesian]')

# Standalone equivalent:
# plot_trace(posterior.draws, posterior.param_names)

In [ ]:
# ---- Credible intervals -----------------------------------------------------
# Equal-tailed 95 % credible intervals for each parameter.

ci_95 = posterior.credible_interval(alpha=0.95)
print('95 % credible intervals:')
for name, (lo, hi) in ci_95.items():
    print(f'  {name:<30s}  [{lo:.4f}, {hi:.4f}]')

# You can request narrower intervals too:
ci_50 = posterior.credible_interval(alpha=0.50)
print('\n50 % credible intervals:')
for name, (lo, hi) in ci_50.items():
    print(f'  {name:<30s}  [{lo:.4f}, {hi:.4f}]')

# Standalone equivalent:
# ci = credible_intervals(posterior.draws, posterior.param_names, alpha=0.95)

In [ ]:
# ---- Gelman-Rubin R‑hat convergence diagnostic ------------------------------
# Requires ``arviz``.  Values close to 1.0 indicate good convergence.

rhat = posterior.gelman_rubin()
if rhat is not None:
    print('Gelman-Rubin R‑hat:')
    for name, r in rhat.items():
        flag = ' ✓' if r < 1.1 else ' ⚠'
        print(f'  {name:<30s}  R̂ = {r:.3f}{flag}')
else:
    print('Skipped: arviz is not installed.')

In [ ]:
# ---- Posterior-predictive reflectivity --------------------------------------
# Propagate parameter uncertainty through the model to obtain median
# reflectivity and 95 % credible band.

r_median, r_lower, r_upper = posterior_predictive_reflectivity(
    posterior.draws, posterior.param_names, model, qz, n_samples=200,
)

plt.figure(figsize=(9, 6))
plt.semilogy(qz, r_data, 'o', label='Data', alpha=0.6)
plt.semilogy(qz, r_median, '-', color='tab:orange', label='Posterior median')
plt.fill_between(
    qz, r_lower, r_upper, color='tab:orange', alpha=0.3,
    label='95 % credible interval',
)
plt.xlabel('Q / Å⁻¹')
plt.ylabel('Reflectivity')
plt.title('Bayesian Posterior-Predictive Check')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(
    'The 95 % credible interval captures parameter uncertainty '
    'propagated through the model.'
)

In [ ]:
# ---- Posterior-predictive SLD profile ---------------------------------------
# The same idea applied to the scattering-length-density profile.

z, sld_median, sld_lower, sld_upper = posterior_predictive_sld_profile(
    posterior.draws, posterior.param_names, model, n_samples=200,
)

plt.figure(figsize=(8, 4))
plt.plot(z, sld_median, label='Posterior median SLD')
plt.fill_between(
    z, sld_lower, sld_upper, alpha=0.3,
    label='95 % credible interval',
)
plt.xlabel('z / Å')
plt.ylabel('SLD / 10⁻⁶ Å⁻²')
plt.title('SLD Profile with Bayesian Uncertainty')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ---- Summary ----------------------------------------------------------------
print('―' * 60)
print('Bayesian analysis complete!')
print('―' * 60)
print()
print('API surface demonstrated:')
print('  MultiFitter.sample(data, samples=, burn=, thin=, seed=)')
print('  PosteriorResults(draws, param_names, logp=, sampler_state=)')
print('    .summary()                — formatted parameter table')
print('    .corner()                 — pairwise correlation plot')
print('    .trace()                  — MCMC chain trace plot')
print('    .credible_interval(alpha) — equal-tailed credible intervals')
print('    .gelman_rubin()           — R̂ convergence diagnostic')
print()
print('  Standalone functions (work on raw dict):')
print('    posterior_summary(draws, names)')
print('    plot_corner(draws, names)')
print('    plot_trace(draws, names)')
print('    credible_intervals(draws, names, alpha)')
print('    posterior_predictive_reflectivity(draws, names, model, q, n)')
print('    posterior_predictive_sld_profile(draws, names, model, n)')
print()
print('The high-level API provides clean, safe access to BUMPS DREAM sampling.')